In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ultranest

In [26]:
data = []
with open("data.txt") as f:
    lines = f.readlines()[36:]
    for line in lines:
        if line[61] == "#":
            continue
        el = line[20:22].strip()
        N = int(line[6:9].strip())
        Z = int(line[11:14].strip())
        A = int(line[16:19].strip())
        binding_energy = float(line[56:67].strip()) * A / 1000
        binding_energy_std = float(line[68:77].strip()) * A / 1000
        data.append([el, N, Z, A, binding_energy, binding_energy_std])
df = pd.DataFrame(data, columns=["Element", "N", "Z", "A", "Binding Energy (MeV)", "Binding Energy std (MeV)"])

In [30]:
df

,Element,N,Z,A,Binding Energy (MeV),Binding Energy std (MeV)
0,n,1,0,1,0.000000,0.000000e+00
1,H,0,1,1,0.000000,0.000000e+00
2,H,1,1,2,2.224566,4.000000e-07
3,H,2,1,3,8.481796,9.000000e-07
4,He,1,2,3,7.718041,3.000000e-07
...,...,...,...,...,...,...
2545,Hs,157,108,265,1933.505561,2.395600e-02
2546,Hs,158,108,266,1941.337453,2.710540e-02
2547,Mt,157,109,266,1934.022107,9.647820e-02
2548,Ds,159,110,269,1950.291722,3.139230e-02


In [43]:
def model_simple(N, Z, A, a_v, a_s, a_c, a_a):
    return a_v*A - a_s*A**(2/3)-a_c*Z*(Z-1)/A**(1/3)-a_a*(N-Z)**2/A

def prior_transform_simple(cube):
    params = np.zeros_like(cube)

    a_v_min = 0.1
    a_v_max = 20.0
    params[0] = 10**(cube[0] * (np.log10(a_v_max) - np.log10(a_v_min)) + np.log10(a_v_min))

    a_s_min = 0.1
    a_s_max = 20.0
    params[1] = 10**(cube[1] * (np.log10(a_s_max) - np.log10(a_s_min)) + np.log10(a_s_min))

    a_c_min = 0.1
    a_c_max = 20.0
    params[2] = 10**(cube[2] * (np.log10(a_c_max) - np.log10(a_c_min)) + np.log10(a_c_min))

    a_a_min = 0.1
    a_a_max = 20.0
    params[3] = 10**(cube[3] * (np.log10(a_a_max) - np.log10(a_a_min)) + np.log10(a_a_min))

    return params

def likelihood_simple(params):
    model = model_simple(df["N"], df["Z"], df["A"].values, *params)

    residual = (df["Binding Energy (MeV)"] - model) / (df["Binding Energy std (MeV)"] + 1e-9)

    log_norm = np.sum(np.log(2 * np.pi * (df["Binding Energy std (MeV)"] + 1e-9)**2))

    return -0.5 * (np.sum(residual**2) + log_norm)

params_simple = ["a_v", "a_s", "a_c", "a_a"]

In [ ]:
sampler_simple = ultranest.ReactiveNestedSampler(
    params_simple, likelihood_simple, prior_transform_simple
)
result_simple = sampler_simple.run()
sampler_simple.print_results()

[ultranest] Sampling 400 live points from prior ...


/usr/lib/python3.14/site-packages/ultranest/integrator.py:1903: UserWarning: Sampling from region seems inefficient (0/40 accepted in iteration 2500). To improve efficiency, modify the transformation so that the current live points are ellipsoidal, or use a stepsampler, or set frac_remain to a lower number (e.g., 0.5) to terminate earlier.
  u, v, logl, nc, quality = self._refill_samples(Lmin, ndraw, nit)
